## Importing the necessary libraries

In [ ]:
import openai
import os

In [ ]:
os.environ['OPENAI_API_KEY'] = ''  # Replace with your actual key


In [ ]:
openai_api_key = os.getenv('OPENAI_API_KEY')
print("OpenAI API Key:", openai_api_key)

In [ ]:
import datasets
from transformers import AutoTokenizer
    
seed = 42

dataset = datasets.load_dataset(
    "../books", split='train'
)

model_path = ''
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


max_length = 512
    
def tokenize_function(examples):
    return tokenizer(examples["context"], padding="max_length", truncation=True, max_length=max_length)

tokenized_datasets = dataset.map(tokenize_function, batched=True)


from sklearn.preprocessing import OneHotEncoder
import numpy as np

train_test_sp = tokenized_datasets.train_test_split(test_size=0.1, shuffle=True, seed=seed)
cols = ["instruction", "category", 'context']
train_val_sp = train_test_sp['train'].train_test_split(test_size=0.2, shuffle=True, seed=seed)
train_data_original = train_val_sp["train"].shuffle(seed=seed)
train_data = train_val_sp["train"].shuffle(seed=seed).map(tokenize_function, remove_columns=cols).rename_column('response', 'label')
val_data = train_val_sp["test"].shuffle(seed=seed).map(tokenize_function, remove_columns=cols).rename_column('response', 'label')
test_data = train_test_sp["test"].shuffle(seed=seed)

print(test_data['instruction'][0])

PE

In [ ]:
style = 'professional'
role = 'literary scholar'
field = 'philosophy'
skill = 'sentiment analysis'

character_prompt = 'You are a {} {} in {}, who is good at {}. '.format(style, role, field, skill)

augmentation_prompt = 'Let us think step by step.'

FS

In [ ]:
fs = '''
There are some examples for your answer: 
'''

indices = {0: [], 1: [], 2: []}

for index, value in enumerate(train_data_ori['response']):
    if len(indices[value]) < 5:
        indices[value].append(index)

for i in indices[0]+indices[1]+indices[2]:
    fs += 'Context: ' + train_data_original['context'][i] + 'Label: ' + train_data_original['response'][i] + '; \n'

Question

In [ ]:
req = 'Just give the answer and do not give any explanation.'

In [ ]:
def generate_content(prompt, fs=None):
    client = openai.OpenAI()
    if fs:
        messages = [
        {"role": "assistant", "content": character_prompt+augmentation_prompt+fs},
        {"role": "user", "content": prompt}
        ]
    else:
        messages = [
        {"role": "assistant", "content": character_prompt+augmentation_prompt},
        {"role": "user", "content": prompt}
        ]
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        temperature=0.1,
        max_tokens=20,
        frequency_penalty=0.0
    )
    response_text = response.choices[0].message.content
    tokens_used = response.usage.total_tokens
    
    return response_text


answers = []
for i in range(len(test_data['instruction'])):
    prompt = str(i) + '  ' + test_data['instruction'][i] + '\n Work: ' + test_data['context'][i] + '\n' + req
    answers.append(int(generate_content(prompt, fs)))

In [ ]:
from sklearn.metrics import confusion_matrix

true_labels = test_data['response']

predicted_labels = answers

# 计算混淆矩阵
cm = confusion_matrix(true_labels, predicted_labels)
cm